In [ ]:
# 검증 데이터 (2024년도) 역별 혼잡도 계산

import pandas as pd

# 각 혼잡도 파일 경로
congestion_files = [
    '/content/24_03_지하철혼잡도.csv',
    '/content/24_06_지하철혼잡도.csv',
    '/content/24_09_지하철혼잡도.csv',
    '/content/24_12_지하철혼잡도.csv'
]

# 파일을 불러오고 '년도' 컬럼 추가 후 하나로 병합
merged_congestion = []

for path in congestion_files:
    try:
        df = pd.read_csv(path, encoding='utf-8')
    except UnicodeDecodeError:
        df = pd.read_csv(path, encoding='cp949')

    # 연도 추출: "24_03_지하철혼잡도.csv" → 2024
    year = 2000 + int(path.split('/')[-1].split('_')[0])
    df['년도'] = year

    # 연번 컬럼이 있는 경우 제거
    if '연번' in df.columns:
        df = df.drop(columns=['연번'])

    merged_congestion.append(df)

# 전체 병합
congestion_2024 = pd.concat(merged_congestion, ignore_index=True)
congestion_2024

# CSV로 저장
congestion_2024.to_csv('2024_지하철혼잡도_통합.csv', index=False, encoding='utf-8-sig')

# 다운로드 링크 생성
from google.colab import files
files.download('2024_지하철혼잡도_통합.csv')

import os, chardet, csv
import pandas as pd, numpy as np

# 저장 디렉토리 생성
output_dir = '/content/congestions'
os.makedirs(output_dir, exist_ok=True)

import zipfile

# 역별로 필터링해서 CSV 저장
saved_files = 0
for station in stations:
    station_df = congestion_df[congestion_df['출발역'] == station]
    if not station_df.empty:
        filename = f"{station}_혼잡도.csv"
        station_df.to_csv(os.path.join(output_dir, filename), index=False, encoding='utf-8-sig')
        saved_files += 1

# ZIP으로 압축
zip_path = '/content/2024_역별_혼잡도.zip'
with zipfile.ZipFile(zip_path, 'w') as zipf:
    for filename in os.listdir(output_dir):
        file_path = os.path.join(output_dir, filename)
        zipf.write(file_path, arcname=filename)
        
# 다운로드 트리거
from google.colab import files
files.download(zip_path)

# 파일 읽어서 DataFrame으로 만들기
bording_df = pd.read_csv('/content/2024_승하차.csv')

print(bording_df.columns.tolist())

# 저장 디렉토리 생성
output_dir = '/content/boardings'
os.makedirs(output_dir, exist_ok=True)

# 역별로 필터링해서 CSV 저장
saved_files = 0
for station in stations:
    station_df = congestion_df[congestion_df['역명'] == station]
    if not station_df.empty:
        filename = f"{station}_승하차.csv"
        station_df.to_csv(os.path.join(output_dir, filename), index=False, encoding='utf-8-sig')
        saved_files += 1

# ZIP으로 압축
zip_path = '/content/2024_역별_승하차_혼잡도.zip'
with zipfile.ZipFile(zip_path, 'w') as zipf:
    for filename in os.listdir(output_dir):
        file_path = os.path.join(output_dir, filename)
        zipf.write(file_path, arcname=filename)
        
# 다운로드 트리거
from google.colab import files
files.download(zip_path)

In [ ]:
stations = ['가락시장','가산디지털단지','강남','강남구청','강동','강동구청','강변','강일','개롱','개화산',
'거여','건대입구','경복궁','경찰병원','고덕','고려대','고속터미널','공덕','공릉','광나루',
'광화문','광흥창','교대','구로디지털단지','구산','구의','구파발','군자','굽은다리','금호',
'길동','길음','김포공항','까치산','낙성대','남구로','남부터미널','남성','남위례','남태령',
'내방','노원','녹번','녹사평','논현','단대오거리','답십리','당고개','당산','대림',
'대청','대치','대흥','도곡','도림천','도봉산','독립문','독바위','돌곶이','동대문',
'동대문역사문화공원','동대입구','동묘앞','동작','둔촌동','디지털미디어시티','뚝섬','뚝섬유원지','마곡','마들',
'마장','마천','마포','마포구청','망원','매봉','먹골','면목','명동','명일',
'모란','목동','몽촌토성','무악재','문래','문정','미사','미아','반포','발산',
'방배','방이','방화','버티고개','보라매','보문','복정','봉천','봉화산','불광',
'사가정','사당','산성','삼각지','삼성','상계','상도','상봉','상수','상왕십리',
'상월곡','상일동','새절','서대문','서울대입구','서울역','서초','석계','석촌','선릉',
'성수','성신여대입구','송정','송파','수락산','수서','수유','수진','숙대입구','숭실대입구',
'시청','신금호','신길','신내','신답','신당','신대방','신대방삼거리','신도림','신림',
'신사','신설동','신용산','신정','신정네거리','신촌','신풍','신흥','쌍문','아차산',
'아현','안국','안암','암사','압구정','애오개','약수','양재','양천구청','양평',
'어린이대공원','여의나루','여의도','역삼','역촌','연신내','영등포구청','영등포시장','오금','오목교',
'옥수','온수','올림픽공원','왕십리','용답','용두','용마산','우장산','월곡','월드컵경기장',
'을지로3가','을지로4가','을지로입구','응암','이대','이촌','이태원','일원','잠실','잠실나루',
'잠실새내','잠원','장승배기','장암','장지','장한평','제기동','종각','종로3가','종로5가',
'종합운동장','중계','중곡','중화','증산','지축','창동','창신','천왕','천호',
'철산','청구','청담','청량리','총신대입구','충무로','충정로','태릉입구','하계','하남검단산',
'하남시청','하남풍산','학동','학여울','한강진','한성대입구','한양대','합정','행당','혜화',
'홍대입구','홍제','화곡','화랑대','회현','효창공원앞']

In [ ]:
# 학습 데이터 (2021~2023년도) 역별 혼잡도 계산
import os, chardet, csv
import pandas as pd, numpy as np

# 호선별 열차 수용 용량 매핑
LINE_CAPACITY = {1:1600, 2:1600, 3:1600, 4:1600, 5:1280, 6:1280, 7:1280, 8:960}
def get_capacity(line):
    return LINE_CAPACITY.get(int(line), 1600)

# 파일 인코딩 탐지
def detect_encoding(path, sample_size=10000):
    raw = open(path, 'rb').read(sample_size)
    return chardet.detect(raw)['encoding']

# 구분자 감지
def detect_delimiter(path, encoding, sample_size=2048):
    txt = open(path, encoding=encoding, errors='ignore').read(sample_size)
    try:
        return csv.Sniffer().sniff(txt, delimiters=[',', '\t', ';', '|']).delimiter
    except:
        return ','

# 역 단위 처리 함수
def process_station(board_path, cong_path, output_path):
    # 0) 파일 로드
    b_enc = detect_encoding(board_path)
    b_sep = detect_delimiter(board_path, b_enc)
    c_enc = detect_encoding(cong_path)
    c_sep = detect_delimiter(cong_path, c_enc)
    bd = pd.read_csv(board_path, encoding=b_enc, sep=b_sep, engine='python')
    cd = pd.read_csv(cong_path, encoding=c_enc, sep=c_sep, engine='python')

    # 1) 컬럼 정리 및 이름 통일
    bd.columns = bd.columns.str.strip()
    cd.columns = cd.columns.str.strip()
    bd = bd.rename(columns={
        'Datetime': '날짜',
        'Line': '호선',
        '구분': '승하차 구분'
    })

    # 2) 호선 컬럼 처리
    bd['호선'] = bd['호선'].astype(str).str.extract(r'(\d+)').astype(int)
    if '호선' not in cd.columns and 'Line' in cd.columns:
        cd['호선'] = cd['Line']
    cd['호선'] = cd['호선'].astype(str).str.extract(r'(\d+)').astype(int)

    # 3) 상하구분 통일
    cd['상하구분'] = cd['상하구분'].replace({'상행': '상선', '하행': '하선'})

    # 4) 승하차 구분 유효성 검사
    bd['승하차 구분'] = bd['승하차 구분'].astype(str).str.strip()
    if not set(['승차','하차']).intersection(bd['승하차 구분'].unique()):
        raise ValueError(f"{board_path}에 '승차' 또는 '하차' 구분이 없습니다")

    # 5) 승하차 데이터 피벗 처리
    time_cols_b = [c for c in bd.columns if ':' in c]
    bl = bd.melt(id_vars=['날짜','호선','승하차 구분'], value_vars=time_cols_b,
                 var_name='time_bin', value_name='count').dropna(subset=['count'])
    bl['hour'] = bl['time_bin'].str.extract(r'(\d+)').astype(int)
    bsum = bl.groupby(['날짜','호선','hour','승하차 구분'], as_index=False)['count'].sum()
    bp = bsum.pivot(index=['날짜','호선','hour'], columns='승하차 구분', values='count')\
             .fillna(0).reset_index()
    for col in ['승차','하차']:
        if col not in bp.columns:
            bp[col] = 0

    # 6) 혼잡도 비율 계산
    time_cols_c = [c for c in cd.columns if ':' in c]
    cl = cd.melt(id_vars=['호선','요일구분','상하구분'], value_vars=time_cols_c,
                 var_name='time_str', value_name='cong').dropna(subset=['cong'])
    cl['hour'] = cl['time_str'].str.extract(r'(\d+)').astype(int)
    cs = cl.groupby(['호선','요일구분','hour','상하구분'], as_index=False)['cong'].sum()
    cp = cs.pivot(index=['호선','요일구분','hour'], columns='상하구분', values='cong')\
           .fillna(0).reset_index()
    cp.columns.name = None

    for col in ['상선', '하선']:
        if col not in cp.columns:
            cp[col] = 0

    cp['inner_ratio'] = np.where((cp['상선'] + cp['하선']) > 0,
                                 cp['상선'] / (cp['상선'] + cp['하선']), 0.5)
    cp['outer_ratio'] = 1 - cp['inner_ratio']

    # 7) 날짜 기반 요일 구분
    bp['date'] = pd.to_datetime(bp['날짜'], format='%Y-%m-%d')
    bp['weekday'] = bp['date'].dt.weekday
    bp['요일구분'] = bp['weekday'].apply(lambda w: '평일' if w < 5 else ('토요일' if w == 5 else '일요일/공휴일'))

    # 8) merge & 계산
    df = pd.merge(bp, cp, on=['호선','요일구분','hour'], how='left')
    df[['inner_ratio','outer_ratio']] = df[['inner_ratio','outer_ratio']].fillna(0.5)
    df['inner_ride']    = (df['승차'] * df['inner_ratio']).round().astype(int)
    df['inner_getoff']  = (df['하차'] * df['inner_ratio']).round().astype(int)
    df['outer_ride']    = (df['승차'] * df['outer_ratio']).round().astype(int)
    df['outer_getoff']  = (df['하차'] * df['outer_ratio']).round().astype(int)

    # 9) 혼잡도 비율
    df['capacity'] = df['호선'].apply(get_capacity)
    df['inner_congestion_pct'] = (df['inner_ride'] / df['capacity'] * 100).round(2)
    df['outer_congestion_pct'] = (df['outer_ride'] / df['capacity'] * 100).round(2)

    # 10) 저장
    out_cols = ['날짜','호선','hour',
                'inner_ride','inner_getoff',
                'outer_ride','outer_getoff',
                'inner_congestion_pct','outer_congestion_pct']
    df[out_cols].to_csv(output_path, index=False, encoding='utf-8-sig')
    print(f" Saved: {output_path}")


# ==== 일괄 처리 예제 ====
BOARD_DIR = '/content/boardings'
CONG_DIR  = '/content/congestions'
OUT_DIR   = '/content/output'
import os; os.makedirs(OUT_DIR, exist_ok=True)

for name in stations:
    board_f = os.path.join(BOARD_DIR, f'{name}_승하차.csv')
    cong_f  = os.path.join(CONG_DIR,  f"{name}_혼잡도.csv")
    out_f   = os.path.join(OUT_DIR,   f'{name}_내외선_승하차_혼잡도.csv')
    process_station(board_f, cong_f, out_f)
    
    
# ZIP으로 압축
zip_path = '/content/2021_2023_역별_승하차_혼잡도.zip'
with zipfile.ZipFile(zip_path, 'w') as zipf:
    for filename in os.listdir(output_dir):
        file_path = os.path.join(output_dir, filename)
        zipf.write(file_path, arcname=filename)
        
# 다운로드 트리거
from google.colab import files
files.download(zip_path)